NOTE: To work with any of our notebooks on Deepnote, you must first
- "Duplicate" the project into your own Deepnote account (see the button in the top right of the GUI), then 
- click "Move this file to notebooks".  

Then it should run!

# Writing (simple) systems in Drake

Drake is a powerful modeling language for authoring models of dynamical systems.  The goal of this exercise is to learn how to author a very simple dynamical systems.  To start, take a few minutes to work through [this Drake tutorial](https://deepnote.com/workspace/Drake-0b3b2c53-a7ad-441b-80f8-bf8350752305/project/Tutorials-2b4fc509-aef2-417d-a40d-6071dfed9199/%2Fdynamical_systems.ipynb).

Your first task is to write a simple *continuous-time* system that implements the dynamics of a simple damped pendulum (with mass, length, and damping set to 1, and gravity set to 10): $$\ddot\theta + \dot\theta + 10\sin\theta = u.$$  Write code into the following block that results in the pendulum_system variable being an *instance* of a Drake System that implements these dynamics using position and velocity as the state.

Hint: You can import `sin` from `numpy` or from `pydrake.math`. Depending on your version of `numpy`, it might print a warning, but this can be safely ignored.

In [4]:
from pydrake.systems.framework import LeafSystem
from pydrake.math import sin
class PendulumSystem(LeafSystem):
    def __init__(self):
        super().__init__()
        self.DeclareContinuousState(2)
        self.DeclareVectorInputPort("u", 1)
        self.DeclareVectorOutputPort("y", 2, self.CalcOutput)
    
    def DoCalcTimeDerivatives(self, context, derivatives):
        x = context.get_continuous_state_vector().CopyToVector()
        theta = x[0]
        theta_dot = x[1]
        
        u = self.get_input_port(0).Eval(context)[0]
        
        theta_ddot = u - theta_dot - 10 * sin(theta)
        derivatives.get_mutable_vector().SetFromVector([theta_dot, theta_ddot])
        
    def CalcOutput(self, context, output):
        x = context.get_continuous_state_vector().CopyToVector()
        output.SetFromVector(x)

pendulum_system = PendulumSystem()
        

Drake [Systems](https://drake.mit.edu/doxygen_cxx/classdrake_1_1systems_1_1_system.html) have methods that define the dynamics, like $$\dot{x} = f(t, x, u), \qquad y = g(t, x, u)$$ for the state dynamics, $f$, and system output function, $g$. Sometimes the equations depend on time, sometimes not. Sometimes they have extra parameters, sometimes not. Not all systems have inputs, and not all systems have states!

To make the input arguments to these methods more succinct, we collect $t, x, u$ and parameters into a single structure, which we call the [Context](https://drake.mit.edu/doxygen_cxx/classdrake_1_1systems_1_1_context.html). Then all of the methods for the dynamical system can be called using that context, e.g.: $$\dot{x} = f(\text{context}), \qquad y = g(\text{context}).$$ We'll use it often throughout the course.

## Autograding
You can check your work by running the following cell:

In [5]:
from underactuated.exercises.grader import Grader
from underactuated.exercises.intro.test_drake_systems import TestDrakeSystems

Grader.grade_output([TestDrakeSystems], [locals()], "results.json")
Grader.print_test_results("results.json")

Total score is 2/2.

Score for test_dynamics (underactuated.exercises.intro.test_drake_systems.TestDrakeSystems.test_dynamics) is 1/1.

Score for test_input_and_state (underactuated.exercises.intro.test_drake_systems.TestDrakeSystems.test_input_and_state) is 1/1.
